# SaT (Segment Any Text) — 가정통신문 1단계 자체화 PoC

**목표**: Claude Haiku 4.5(외부 API, 8-16s)를 SaT(자체 모델, ~1-5s 예상)로 대체 가능한지 검증.

**SaT 핵심**:
- 텍스트만 입력 (bbox/image 불필요) — pdfplumber 출력 그대로 받음
- 다국어 사전학습 (한국어 포함)
- poorly formatted text 강건 (가정통신문 표·괄호·줄바꿈 많음)
- 학습 데이터 불필요 (drop-in replacement)

**검증 항목**:
1. 한국어 문장 경계 정확성
2. 표 셀 안 짧은 조각 보존
3. 괄호·번호 항목 보존
4. 추론 시간 (Claude vs SaT)
5. bbox 후처리 보강 가능성

**참고**: 발표 §3-4 자체화 시도 흔적의 후속 PoC. 이전 LiLT(셔플 43.3%), LayoutXLM(train/test 9.7%) 결과는 `sentence_extraction/README.md` 참고.

## 1. 환경 설치

- `wtpsplit`: SaT 공식 라이브러리
- `pdfplumber`: PDF text + bbox 추출 (우리 backend와 동일)

In [ ]:
!pip install -q wtpsplit pdfplumber

## 2. SaT 모델 로드

옵션:
- `sat-3l-sm` — 3 layer small (가장 빠름, NCP CPU 최적, ~25M params)
- `sat-12l-sm` — 12 layer small (가장 정밀, 본 PoC 기본)
- `sat-12l` — 12 layer full (최고 정밀, 큰 사이즈)

코랩 GPU 있으면 `sat-12l-sm` 추천. NCP 배포용으론 `sat-3l-sm`도 같이 측정.

In [ ]:
from wtpsplit import SaT
import time

# 두 모델 다 로드해서 비교
print('Loading sat-3l-sm (lightweight)...')
t0 = time.time()
sat_sm = SaT('sat-3l-sm')
print(f'  loaded in {time.time()-t0:.1f}s')

print('Loading sat-12l-sm (accurate)...')
t0 = time.time()
sat_lg = SaT('sat-12l-sm')
print(f'  loaded in {time.time()-t0:.1f}s')

# 빠른 동작 확인
test = '안녕하세요. 오늘은 학부모님께 안내드립니다. 1) 준비물: 필통, 공책. 2) 일시: 5월 20일.'
print('\nQuick check (sat-12l-sm):')
for s in sat_lg.split(test):
    print(f'  | {s}')

## 3. 가정통신문 PDF 업로드

`backend/data/` 안 가정통신문 PDF 1-6장을 업로드하세요. 이전 PoC에서 사용한 6장:
- `2024학년도 4학년 현장체험학습 정산 안내.pdf`
- `2025. 겨울방학 도서관 이용 및 독서캠프 신청 안내.pdf`
- `2026 2,3,5,6학년 구강검진 실시안내.pdf`
- `2026 북부과학교육관 어린이날 행사 안내 가정통신문.pdf`
- `2026년+5월+서귀포외국문화학습관+토요프로그램+추가+모집+안내.pdf`
- `인플루엔자예방접종접종안내(2019).pdf`

In [ ]:
from google.colab import files
uploaded = files.upload()
print(f'\n업로드된 파일 {len(uploaded)}개:')
for fname in uploaded.keys():
    print(f'  - {fname} ({len(uploaded[fname])//1024} KB)')

## 4. pdfplumber 텍스트 추출

우리 backend `app/services/parser.py`와 동일한 방식 (`use_text_flow=True`, `x_tolerance=3`, `y_tolerance=3`).
이전 LayoutXLM PoC에서 검증된 옵션.

In [ ]:
import pdfplumber

def extract_text(pdf_path: str) -> str:
    """backend와 동일한 방식으로 PDF 텍스트 추출."""
    parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text(use_text_flow=True, x_tolerance=3, y_tolerance=3) or ''
            parts.append(txt)
    return '\n'.join(parts)

texts = {}
for fname in uploaded.keys():
    txt = extract_text(fname)
    texts[fname] = txt
    print(f'{fname}: {len(txt)} 자')

## 5. SaT 문장 분할 (메인 검증)

두 모델(`sat-3l-sm`, `sat-12l-sm`) 비교 + 추론 시간 측정.

In [ ]:
results = {}

for fname, txt in texts.items():
    print(f'\n========== {fname} ==========')
    print(f'원본: {len(txt)} 자')
    print(f'원본 첫 200자: {txt[:200]!r}')
    
    # sat-3l-sm (lightweight)
    t0 = time.time()
    sents_sm = list(sat_sm.split(txt))
    t_sm = time.time() - t0
    
    # sat-12l-sm (accurate)
    t0 = time.time()
    sents_lg = list(sat_lg.split(txt))
    t_lg = time.time() - t0
    
    print(f'\nsat-3l-sm  : {len(sents_sm)} 문장, {t_sm:.2f}s')
    print(f'sat-12l-sm : {len(sents_lg)} 문장, {t_lg:.2f}s')
    
    results[fname] = {
        'sm': {'sentences': sents_sm, 'time': t_sm},
        'lg': {'sentences': sents_lg, 'time': t_lg},
    }

## 6. 결과 비교 — sat-12l-sm 상세 출력

한국어 문장 경계, 표 처리, 괄호·번호 보존을 눈으로 검증.

In [ ]:
for fname, r in results.items():
    print(f'\n========== {fname} ==========')
    print(f'[sat-12l-sm] 문장 {len(r["lg"]["sentences"])}개:\n')
    for i, s in enumerate(r['lg']['sentences']):
        s_clean = s.strip().replace('\n', ' | ')
        if len(s_clean) > 120:
            s_clean = s_clean[:120] + '...'
        print(f'  [{i:02d}] {s_clean}')

## 7. 변형 0 보장 검증 — span 보존

SaT가 텍스트를 변형 없이 분할하는지(generation이 아닌지) 확인. `''.join(sentences) == original` 또는 거의 동일해야 함.

In [ ]:
for fname, r in results.items():
    txt = texts[fname]
    joined_lg = ''.join(r['lg']['sentences'])
    
    # 정확히 동일?
    exact = txt == joined_lg
    
    # 공백/줄바꿈만 차이?
    norm_txt = ''.join(txt.split())
    norm_joined = ''.join(joined_lg.split())
    char_preserved = norm_txt == norm_joined
    
    print(f'{fname}:')
    print(f'  exact match: {exact}')
    print(f'  char preserved (공백 무시): {char_preserved}')
    if not char_preserved:
        # 길이 차이만
        print(f'  원본 char: {len(norm_txt)}, joined char: {len(norm_joined)}, diff: {len(norm_txt) - len(norm_joined)}')

## 8. 추론 시간 — Claude Haiku 4.5 vs SaT

메모리 기준 Claude Haiku 4.5는 8~16초/문서. SaT는 코랩 GPU에서 측정.
NCP CPU 추론은 별도 측정 필요 (보통 GPU 대비 5-10배 느림).

In [ ]:
import statistics

sm_times = [r['sm']['time'] for r in results.values()]
lg_times = [r['lg']['time'] for r in results.values()]

print('=== 추론 시간 비교 ===')
print(f'sat-3l-sm  : mean {statistics.mean(sm_times):.2f}s, max {max(sm_times):.2f}s')
print(f'sat-12l-sm : mean {statistics.mean(lg_times):.2f}s, max {max(lg_times):.2f}s')
print(f'Claude Haiku 4.5 (메모리 기준): 8-16s')
print()
print('※ 코랩 GPU 측정. NCP CPU에선 5-10배 느려질 수 있음.')
print('  sat-3l-sm이 NCP CPU에서도 3-5s 안에 들어올 가능성 큼.')

## 9. (선택) bbox 후처리 — 표 셀 보존 보강

SaT는 text-only라 표 셀을 깰 가능성이 있음. pdfplumber로 bbox도 같이 받아서, **같은 y좌표(행) 토큰을 묶어** 표 행 단위로 보존하는 후처리를 시도.

이 셀은 SaT 단독 결과에 만족 못 할 때 활용. 충분하면 skip.

In [ ]:
def extract_with_bbox(pdf_path: str):
    """같은 y좌표(±tolerance) 토큰을 같은 행으로 묶기."""
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words(use_text_flow=True, x_tolerance=3, y_tolerance=3)
            # y0 기준 묶기 (tolerance 2pt)
            words.sort(key=lambda w: (w['top'], w['x0']))
            cur_y = None
            cur_row = []
            for w in words:
                if cur_y is None or abs(w['top'] - cur_y) > 2:
                    if cur_row:
                        rows.append(' '.join(t['text'] for t in cur_row))
                    cur_row = [w]
                    cur_y = w['top']
                else:
                    cur_row.append(w)
            if cur_row:
                rows.append(' '.join(t['text'] for t in cur_row))
    return rows

# 첫 PDF에 대해 데모
first_pdf = list(uploaded.keys())[0]
rows = extract_with_bbox(first_pdf)
print(f'{first_pdf}: bbox 행 기반 {len(rows)} 행')
for i, r in enumerate(rows[:30]):
    print(f'  [{i:02d}] {r[:100]}')

## 10. 결론 정리

PoC 평가 체크리스트:
- [ ] 한국어 문장 경계 정확한가? ("입니다.", "습니다." 등 종결 인식)
- [ ] 표 셀 안 짧은 조각 (날짜·금액·이름) 보존되나? 
- [ ] 번호 항목(1), 2), 3))이 한 문장으로 묶이나, 잘리나?
- [ ] 괄호 설명(예: "체험비 18,000원 (1인당)")이 보존되나?
- [ ] 추론 시간이 Claude 8-16s 대비 우위인가?
- [ ] 변형 0 보장(char preserved) 통과하나?

**결과 해석**:
- 위 6개 다 통과 → SaT 단독으로 자체화 가능. NCP CPU 추론 시간 측정 후 배포 검토.
- 한국어 종결/괄호는 OK인데 표 셀만 약함 → SaT + bbox 후처리 하이브리드.
- 한국어 종결 자체가 약함 → 한국어 fine-tuning 또는 다른 모델(LayoutLMv3 + BIO 라벨) 필요.

**다음 단계**:
- 결과 좋으면 → `backend/app/services/` 안에 SaT 추론 모듈 추가, Claude API 대체.
- 한계 보이면 → bbox 후처리 추가 또는 LayoutLMv3 BIO 학습으로 확장.